## Purpose: Define the feature-engineering workflow for purchase prediction and personalized recommendations.

# 02 — Feature Engineering

In this notebook, we transform the cleaned Retailrocket event data into behavior-based features for purchase prediction and personalized recommendation.

**Main objectives:**
- Load the cleaned event data
- Create temporal features from event timestamps
- Construct user-item behavioral features
- Define a time-aware purchase-prediction target
- Prevent information leakage by using past behavior only
- Save the final modeling dataset for subsequent notebooks

In [2]:
# Purpose: Import libraries and configure display settings for feature engineering.

from pathlib import Path
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 120

RANDOM_STATE = 42

In [3]:
# Purpose: Locate the project root automatically and load the cleaned event dataset.

current_path = Path.cwd().resolve()

project_root = next(
    (
        candidate
        for candidate in [current_path, *current_path.parents]
        if (candidate / "data").exists()
    ),
    current_path
)

processed_data_dir = project_root / "data" / "processed"
figures_dir = project_root / "outputs" / "figures"
tables_dir = project_root / "outputs" / "tables"

cleaned_events_path = processed_data_dir / "events_cleaned.csv"

print(f"Project root: {project_root}")
print(f"Cleaned file exists: {cleaned_events_path.exists()}")

events_df = pd.read_csv(cleaned_events_path)

print(f"\nCleaned events dataset shape: {events_df.shape}")
display(events_df.head())

Project root: F:\ecommerce-recommender-xai
Cleaned file exists: True

Cleaned events dataset shape: (2755641, 5)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [4]:
# Purpose: Convert timestamps to datetime values and sort all events in chronological order.

events_df["event_datetime"] = pd.to_datetime(
    events_df["timestamp"],
    unit="ms"
)

events_df = (
    events_df
    .sort_values("event_datetime")
    .reset_index(drop=True)
)

print(f"First event: {events_df['event_datetime'].min()}")
print(f"Last event: {events_df['event_datetime'].max()}")

display(
    events_df[
        ["event_datetime", "visitorid", "itemid", "event", "transactionid"]
    ].head()
)

First event: 2015-05-03 03:00:04.384000
Last event: 2015-09-18 02:59:47.788000


,event_datetime,visitorid,itemid,event,transactionid
0,2015-05-03 03:00:04.384,693516,297662,addtocart,NaN
1,2015-05-03 03:00:11.289,829044,60987,view,NaN
2,2015-05-03 03:00:13.048,652699,252860,view,NaN
3,2015-05-03 03:00:24.154,1125936,33661,view,NaN
4,2015-05-03 03:00:26.228,693516,297662,view,NaN


In [5]:
# Purpose: Define a chronological cutoff so features use past events and labels use only future purchases.

time_cutoff = events_df["event_datetime"].quantile(0.80)

history_events = events_df[
    events_df["event_datetime"] < time_cutoff
].copy()

future_events = events_df[
    events_df["event_datetime"] >= time_cutoff
].copy()

print(f"Time cutoff: {time_cutoff}")
print(f"\nHistory events: {len(history_events):,}")
print(f"Future events: {len(future_events):,}")
print(
    f"History period: "
    f"{history_events['event_datetime'].min()} to "
    f"{history_events['event_datetime'].max()}"
)
print(
    f"Future period: "
    f"{future_events['event_datetime'].min()} to "
    f"{future_events['event_datetime'].max()}"
)

Time cutoff: 2015-08-18 04:23:20.444999936

History events: 2,204,512
Future events: 551,129
History period: 2015-05-03 03:00:04.384000 to 2015-08-18 04:23:01.129000
Future period: 2015-08-18 04:23:20.445000 to 2015-09-18 02:59:47.788000


In [6]:
# Purpose: Identify unique user-item pairs that completed at least one transaction after the time cutoff.

future_purchase_pairs = (
    future_events[
        future_events["event"] == "transaction"
    ][["visitorid", "itemid"]]
    .drop_duplicates()
    .copy()
)

future_purchase_pairs["future_purchase"] = 1

print(f"Future transaction records: {(future_events['event'] == 'transaction').sum():,}")
print(f"Unique user-item pairs with a future purchase: {len(future_purchase_pairs):,}")

display(future_purchase_pairs.head())

Future transaction records: 4,593
Unique user-item pairs with a future purchase: 4,382


,visitorid,itemid,future_purchase
2204650,873157,252564,1
2204651,873157,110512,1
2204764,1183970,463052,1
2205031,19512,272880,1
2205180,803960,440937,1


In [7]:
# Purpose: Aggregate past user-item behavior into event counts and interaction-time features.

history_pair_features = (
    history_events
    .groupby(["visitorid", "itemid"])
    .agg(
        total_interactions=("event", "size"),
        first_interaction_time=("event_datetime", "min"),
        last_interaction_time=("event_datetime", "max")
    )
    .join(
        history_events
        .groupby(["visitorid", "itemid", "event"])
        .size()
        .unstack(fill_value=0),
        how="left"
    )
    .reset_index()
)

for event_type in ["view", "addtocart", "transaction"]:
    if event_type not in history_pair_features.columns:
        history_pair_features[event_type] = 0

history_pair_features = history_pair_features[
    [
        "visitorid",
        "itemid",
        "total_interactions",
        "view",
        "addtocart",
        "transaction",
        "first_interaction_time",
        "last_interaction_time"
    ]
]

print("Historical user-item feature table shape:", history_pair_features.shape)
display(history_pair_features.head())

Historical user-item feature table shape: (1713171, 8)


,visitorid,itemid,total_interactions,view,addtocart,transaction,first_interaction_time,last_interaction_time
0,1,72028,1,1,0,0,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444
1,2,216305,2,2,0,0,2015-08-07 18:01:08.920,2015-08-07 18:17:43.170
2,2,259884,1,1,0,0,2015-08-07 17:56:52.664,2015-08-07 17:56:52.664
3,2,325215,3,3,0,0,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845
4,2,342816,2,2,0,0,2015-08-07 18:08:25.669,2015-08-07 18:17:24.375


In [8]:
# Purpose: Assign the future-purchase target to historical user-item pairs without using future behavior as a feature.

modeling_pairs = history_pair_features.merge(
    future_purchase_pairs,
    on=["visitorid", "itemid"],
    how="left"
)

modeling_pairs["future_purchase"] = (
    modeling_pairs["future_purchase"]
    .fillna(0)
    .astype(int)
)

known_future_purchase_pairs = modeling_pairs["future_purchase"].sum()
unseen_future_purchase_pairs = (
    len(future_purchase_pairs) - known_future_purchase_pairs
)

print(f"Historical user-item pairs available for modeling: {len(modeling_pairs):,}")
print(f"Future purchases linked to a historical pair: {known_future_purchase_pairs:,}")
print(f"Future purchases with no historical user-item interaction: {unseen_future_purchase_pairs:,}")
print(
    f"Future purchase rate among historical pairs: "
    f"{modeling_pairs['future_purchase'].mean() * 100:.4f}%"
)

display(modeling_pairs.head())

Historical user-item pairs available for modeling: 1,713,171
Future purchases linked to a historical pair: 186
Future purchases with no historical user-item interaction: 4,196
Future purchase rate among historical pairs: 0.0109%


,visitorid,itemid,total_interactions,view,addtocart,transaction,first_interaction_time,last_interaction_time,future_purchase
0,1,72028,1,1,0,0,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,0
1,2,216305,2,2,0,0,2015-08-07 18:01:08.920,2015-08-07 18:17:43.170,0
2,2,259884,1,1,0,0,2015-08-07 17:56:52.664,2015-08-07 17:56:52.664,0
3,2,325215,3,3,0,0,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,0
4,2,342816,2,2,0,0,2015-08-07 18:08:25.669,2015-08-07 18:17:24.375,0


In [9]:
# Purpose: Define a user-level future-purchase target to obtain a viable prediction dataset.

future_purchasing_users = (
    future_events[
        future_events["event"] == "transaction"
    ][["visitorid"]]
    .drop_duplicates()
    .copy()
)

future_purchasing_users["future_purchase_user"] = 1

history_users = (
    history_events[["visitorid"]]
    .drop_duplicates()
    .copy()
)

user_target_table = history_users.merge(
    future_purchasing_users,
    on="visitorid",
    how="left"
)

user_target_table["future_purchase_user"] = (
    user_target_table["future_purchase_user"]
    .fillna(0)
    .astype(int)
)

print(f"Users with historical activity: {len(user_target_table):,}")
print(f"Users with a future purchase: {user_target_table['future_purchase_user'].sum():,}")
print(
    f"Future user-level purchase rate: "
    f"{user_target_table['future_purchase_user'].mean() * 100:.3f}%"
)

Users with historical activity: 1,123,765
Users with a future purchase: 352
Future user-level purchase rate: 0.031%


In [10]:
# Purpose: Define cart-to-purchase candidates and label whether the same item is purchased within 14 days.

purchase_horizon_days = 14

first_cart_events = (
    events_df[events_df["event"] == "addtocart"]
    .sort_values("event_datetime")
    .drop_duplicates(subset=["visitorid", "itemid"], keep="first")
    [["visitorid", "itemid", "event_datetime"]]
    .rename(columns={"event_datetime": "cart_time"})
    .copy()
)

first_transaction_events = (
    events_df[events_df["event"] == "transaction"]
    .sort_values("event_datetime")
    .groupby(["visitorid", "itemid"], as_index=False)["event_datetime"]
    .min()
    .rename(columns={"event_datetime": "first_purchase_time"})
)

cart_candidates = first_cart_events.merge(
    first_transaction_events,
    on=["visitorid", "itemid"],
    how="left"
)

cart_candidates["purchased_within_14_days"] = (
    cart_candidates["first_purchase_time"].notna()
    & (
        cart_candidates["first_purchase_time"]
        > cart_candidates["cart_time"]
    )
    & (
        cart_candidates["first_purchase_time"]
        <= cart_candidates["cart_time"] + pd.Timedelta(days=purchase_horizon_days)
    )
).astype(int)

print(f"Unique user-item cart candidates: {len(cart_candidates):,}")
print(
    f"Positive cart-to-purchase cases: "
    f"{cart_candidates['purchased_within_14_days'].sum():,}"
)
print(
    f"Cart-to-purchase rate within {purchase_horizon_days} days: "
    f"{cart_candidates['purchased_within_14_days'].mean() * 100:.2f}%"
)

display(cart_candidates.head())

Unique user-item cart candidates: 62,025
Positive cart-to-purchase cases: 18,864
Cart-to-purchase rate within 14 days: 30.41%


,visitorid,itemid,cart_time,first_purchase_time,purchased_within_14_days
0,693516,297662,2015-05-03 03:00:04.384,NaT,0
1,979664,338222,2015-05-03 03:01:25.008,NaT,0
2,260113,125751,2015-05-03 03:04:00.603,NaT,0
3,319455,342530,2015-05-03 03:10:02.006,NaT,0
4,345781,438400,2015-05-03 03:11:33.415,2015-05-03 03:35:01.772,1


In [11]:
# Purpose: Remove late cart events that do not have a complete 14-day observation window for purchase labeling.

dataset_end_time = events_df["event_datetime"].max()

last_eligible_cart_time = (
    dataset_end_time - pd.Timedelta(days=purchase_horizon_days)
)

cart_modeling_data = cart_candidates[
    cart_candidates["cart_time"] <= last_eligible_cart_time
].copy()

print(f"Dataset end time: {dataset_end_time}")
print(f"Last eligible cart time: {last_eligible_cart_time}")
print(f"\nEligible cart candidates: {len(cart_modeling_data):,}")
print(
    f"Positive cart-to-purchase cases: "
    f"{cart_modeling_data['purchased_within_14_days'].sum():,}"
)
print(
    f"Cart-to-purchase rate within {purchase_horizon_days} days: "
    f"{cart_modeling_data['purchased_within_14_days'].mean() * 100:.2f}%"
)

Dataset end time: 2015-09-18 02:59:47.788000
Last eligible cart time: 2015-09-04 02:59:47.788000

Eligible cart candidates: 56,385
Positive cart-to-purchase cases: 17,310
Cart-to-purchase rate within 14 days: 30.70%


In [12]:
# Purpose: Extract and aggregate user-item events that occurred strictly before the first cart event.

cart_keys = cart_modeling_data[
    ["visitorid", "itemid", "cart_time"]
].copy()

pre_cart_pair_events = events_df.merge(
    cart_keys,
    on=["visitorid", "itemid"],
    how="inner"
)

pre_cart_pair_events = pre_cart_pair_events[
    pre_cart_pair_events["event_datetime"]
    < pre_cart_pair_events["cart_time"]
].copy()

pre_cart_pair_features = (
    pre_cart_pair_events
    .groupby(["visitorid", "itemid", "cart_time"])
    .agg(
        prior_pair_event_count=("event", "size"),
        first_prior_pair_event_time=("event_datetime", "min"),
        last_prior_pair_event_time=("event_datetime", "max")
    )
    .join(
        pre_cart_pair_events
        .groupby(["visitorid", "itemid", "cart_time", "event"])
        .size()
        .unstack(fill_value=0),
        how="left"
    )
    .reset_index()
)

for event_type in ["view", "addtocart", "transaction"]:
    if event_type not in pre_cart_pair_features.columns:
        pre_cart_pair_features[event_type] = 0

pre_cart_pair_features = pre_cart_pair_features.rename(columns={
    "view": "prior_pair_view_count",
    "addtocart": "prior_pair_cart_count",
    "transaction": "prior_pair_transaction_count"
})

print(f"Cart candidates with at least one prior same-pair event: {len(pre_cart_pair_features):,}")
display(pre_cart_pair_features.head())

Cart candidates with at least one prior same-pair event: 41,620


,visitorid,itemid,cart_time,prior_pair_event_count,first_prior_pair_event_time,last_prior_pair_event_time,prior_pair_transaction_count,prior_pair_view_count,prior_pair_cart_count
0,150,452955,2015-06-07 23:30:18.230,1,2015-06-07 23:29:00.599,2015-06-07 23:29:00.599,0,1,0
1,172,10034,2015-08-15 00:50:16.912,5,2015-07-14 04:43:09.537,2015-08-15 00:49:12.998,0,5,0
2,172,465522,2015-08-15 01:13:39.691,1,2015-08-15 01:06:38.438,2015-08-15 01:06:38.438,0,1,0
3,299,149253,2015-06-22 18:19:52.866,1,2015-06-22 18:04:27.907,2015-06-22 18:04:27.907,0,1,0
4,318,133814,2015-06-29 19:56:25.773,1,2015-06-29 19:54:50.724,2015-06-29 19:54:50.724,0,1,0


In [13]:
# Purpose: Combine pre-cart features with the purchase target and derive recency and timing features.

cart_feature_table = cart_modeling_data.merge(
    pre_cart_pair_features,
    on=["visitorid", "itemid", "cart_time"],
    how="left"
)

count_feature_columns = [
    "prior_pair_event_count",
    "prior_pair_view_count",
    "prior_pair_cart_count",
    "prior_pair_transaction_count"
]

cart_feature_table[count_feature_columns] = (
    cart_feature_table[count_feature_columns]
    .fillna(0)
    .astype(int)
)

cart_feature_table["has_prior_pair_interaction"] = (
    cart_feature_table["prior_pair_event_count"] > 0
).astype(int)

cart_feature_table["prior_pair_recency_minutes"] = (
    cart_feature_table["cart_time"]
    - cart_feature_table["last_prior_pair_event_time"]
).dt.total_seconds() / 60

cart_feature_table["prior_pair_interaction_span_minutes"] = (
    cart_feature_table["cart_time"]
    - cart_feature_table["first_prior_pair_event_time"]
).dt.total_seconds() / 60

cart_feature_table["cart_hour"] = cart_feature_table["cart_time"].dt.hour
cart_feature_table["cart_dayofweek"] = cart_feature_table["cart_time"].dt.dayofweek
cart_feature_table["cart_month"] = cart_feature_table["cart_time"].dt.month

print("Cart-level feature table shape:", cart_feature_table.shape)

display(
    cart_feature_table[
        [
            "visitorid",
            "itemid",
            "cart_time",
            "prior_pair_event_count",
            "prior_pair_view_count",
            "has_prior_pair_interaction",
            "prior_pair_recency_minutes",
            "prior_pair_interaction_span_minutes",
            "purchased_within_14_days"
        ]
    ].head()
)

Cart-level feature table shape: (56385, 17)


,visitorid,itemid,cart_time,prior_pair_event_count,prior_pair_view_count,has_prior_pair_interaction,prior_pair_recency_minutes,prior_pair_interaction_span_minutes,purchased_within_14_days
0,693516,297662,2015-05-03 03:00:04.384,0,0,0,NaN,NaN,0
1,979664,338222,2015-05-03 03:01:25.008,0,0,0,NaN,NaN,0
2,260113,125751,2015-05-03 03:04:00.603,1,1,1,2.877,2.877,0
3,319455,342530,2015-05-03 03:10:02.006,0,0,0,NaN,NaN,0
4,345781,438400,2015-05-03 03:11:33.415,1,1,1,2.088,2.088,1


In [14]:
# Purpose: Calculate each user's cumulative behavior strictly before the cart event time.

user_event_history = events_df[
    ["visitorid", "event_datetime", "event"]
].copy()

user_event_history["user_event_count_before_cart"] = (
    user_event_history
    .groupby("visitorid")
    .cumcount()
    + 1
)

for event_type in ["view", "addtocart", "transaction"]:
    user_event_history[f"user_{event_type}_count_before_cart"] = (
        user_event_history["event"].eq(event_type).astype(int)
        .groupby(user_event_history["visitorid"])
        .cumsum()
    )

user_event_history = user_event_history[
    [
        "visitorid",
        "event_datetime",
        "user_event_count_before_cart",
        "user_view_count_before_cart",
        "user_addtocart_count_before_cart",
        "user_transaction_count_before_cart"
    ]
].sort_values("event_datetime")

cart_feature_table = pd.merge_asof(
    cart_feature_table.sort_values("cart_time"),
    user_event_history,
    left_on="cart_time",
    right_on="event_datetime",
    by="visitorid",
    direction="backward",
    allow_exact_matches=False
).drop(columns="event_datetime")

user_history_feature_columns = [
    "user_event_count_before_cart",
    "user_view_count_before_cart",
    "user_addtocart_count_before_cart",
    "user_transaction_count_before_cart"
]

cart_feature_table[user_history_feature_columns] = (
    cart_feature_table[user_history_feature_columns]
    .fillna(0)
    .astype(int)
)

cart_feature_table = cart_feature_table.sort_values(
    ["cart_time", "visitorid", "itemid"]
).reset_index(drop=True)

display(
    cart_feature_table[
        [
            "visitorid",
            "itemid",
            "cart_time",
            "user_event_count_before_cart",
            "user_view_count_before_cart",
            "user_addtocart_count_before_cart",
            "user_transaction_count_before_cart",
            "purchased_within_14_days"
        ]
    ].head()
)

,visitorid,itemid,cart_time,user_event_count_before_cart,user_view_count_before_cart,user_addtocart_count_before_cart,user_transaction_count_before_cart,purchased_within_14_days
0,693516,297662,2015-05-03 03:00:04.384,0,0,0,0,0
1,979664,338222,2015-05-03 03:01:25.008,0,0,0,0,0
2,260113,125751,2015-05-03 03:04:00.603,2,2,0,0,0
3,319455,342530,2015-05-03 03:10:02.006,0,0,0,0,0
4,345781,438400,2015-05-03 03:11:33.415,1,1,0,0,1


In [15]:
# Purpose: Calculate each item's cumulative behavior strictly before the cart event time.

item_event_history = events_df[
    ["itemid", "event_datetime", "event"]
].copy()

item_event_history["item_event_count_before_cart"] = (
    item_event_history
    .groupby("itemid")
    .cumcount()
    + 1
)

for event_type in ["view", "addtocart", "transaction"]:
    item_event_history[f"item_{event_type}_count_before_cart"] = (
        item_event_history["event"].eq(event_type).astype(int)
        .groupby(item_event_history["itemid"])
        .cumsum()
    )

item_event_history = item_event_history[
    [
        "itemid",
        "event_datetime",
        "item_event_count_before_cart",
        "item_view_count_before_cart",
        "item_addtocart_count_before_cart",
        "item_transaction_count_before_cart"
    ]
].sort_values("event_datetime")

cart_feature_table = pd.merge_asof(
    cart_feature_table.sort_values("cart_time"),
    item_event_history,
    left_on="cart_time",
    right_on="event_datetime",
    by="itemid",
    direction="backward",
    allow_exact_matches=False
).drop(columns="event_datetime")

item_history_feature_columns = [
    "item_event_count_before_cart",
    "item_view_count_before_cart",
    "item_addtocart_count_before_cart",
    "item_transaction_count_before_cart"
]

cart_feature_table[item_history_feature_columns] = (
    cart_feature_table[item_history_feature_columns]
    .fillna(0)
    .astype(int)
)

cart_feature_table = cart_feature_table.sort_values(
    ["cart_time", "visitorid", "itemid"]
).reset_index(drop=True)

display(
    cart_feature_table[
        [
            "visitorid",
            "itemid",
            "cart_time",
            "item_event_count_before_cart",
            "item_view_count_before_cart",
            "item_addtocart_count_before_cart",
            "item_transaction_count_before_cart",
            "purchased_within_14_days"
        ]
    ].head()
)

,visitorid,itemid,cart_time,item_event_count_before_cart,item_view_count_before_cart,item_addtocart_count_before_cart,item_transaction_count_before_cart,purchased_within_14_days
0,693516,297662,2015-05-03 03:00:04.384,0,0,0,0,0
1,979664,338222,2015-05-03 03:01:25.008,0,0,0,0,0
2,260113,125751,2015-05-03 03:04:00.603,1,1,0,0,0
3,319455,342530,2015-05-03 03:10:02.006,0,0,0,0,0
4,345781,438400,2015-05-03 03:11:33.415,1,1,0,0,1


In [16]:
# Purpose: Derive safe ratio-based features from prior user, item, and user-item behavior.

cart_feature_table["user_prior_purchase_share"] = np.divide(
    cart_feature_table["user_transaction_count_before_cart"],
    cart_feature_table["user_event_count_before_cart"],
    out=np.zeros(len(cart_feature_table), dtype=float),
    where=cart_feature_table["user_event_count_before_cart"] > 0
)

cart_feature_table["user_prior_cart_share"] = np.divide(
    cart_feature_table["user_addtocart_count_before_cart"],
    cart_feature_table["user_event_count_before_cart"],
    out=np.zeros(len(cart_feature_table), dtype=float),
    where=cart_feature_table["user_event_count_before_cart"] > 0
)

cart_feature_table["item_prior_purchase_share"] = np.divide(
    cart_feature_table["item_transaction_count_before_cart"],
    cart_feature_table["item_event_count_before_cart"],
    out=np.zeros(len(cart_feature_table), dtype=float),
    where=cart_feature_table["item_event_count_before_cart"] > 0
)

cart_feature_table["item_prior_cart_share"] = np.divide(
    cart_feature_table["item_addtocart_count_before_cart"],
    cart_feature_table["item_event_count_before_cart"],
    out=np.zeros(len(cart_feature_table), dtype=float),
    where=cart_feature_table["item_event_count_before_cart"] > 0
)

cart_feature_table["prior_pair_view_share"] = np.divide(
    cart_feature_table["prior_pair_view_count"],
    cart_feature_table["prior_pair_event_count"],
    out=np.zeros(len(cart_feature_table), dtype=float),
    where=cart_feature_table["prior_pair_event_count"] > 0
)

display(
    cart_feature_table[
        [
            "user_prior_purchase_share",
            "user_prior_cart_share",
            "item_prior_purchase_share",
            "item_prior_cart_share",
            "prior_pair_view_share",
            "purchased_within_14_days"
        ]
    ].head()
)

,user_prior_purchase_share,user_prior_cart_share,item_prior_purchase_share,item_prior_cart_share,prior_pair_view_share,purchased_within_14_days
0,0.000,0.000,0.000,0.000,0.000,0
1,0.000,0.000,0.000,0.000,0.000,0
2,0.000,0.000,0.000,0.000,1.000,0
3,0.000,0.000,0.000,0.000,0.000,0
4,0.000,0.000,0.000,0.000,1.000,1


In [17]:
# Purpose: Reduce skew in count features and explicitly flag missing same-pair history.

count_features_for_log = [
    "prior_pair_event_count",
    "prior_pair_view_count",
    "prior_pair_cart_count",
    "prior_pair_transaction_count",
    "user_event_count_before_cart",
    "user_view_count_before_cart",
    "user_addtocart_count_before_cart",
    "user_transaction_count_before_cart",
    "item_event_count_before_cart",
    "item_view_count_before_cart",
    "item_addtocart_count_before_cart",
    "item_transaction_count_before_cart"
]

for feature in count_features_for_log:
    cart_feature_table[f"log_{feature}"] = np.log1p(
        cart_feature_table[feature]
    )

cart_feature_table["missing_prior_pair_timing"] = (
    cart_feature_table["last_prior_pair_event_time"].isna()
).astype(int)

print("Number of model features created so far:", cart_feature_table.shape[1])

display(
    cart_feature_table[
        [
            "prior_pair_event_count",
            "log_prior_pair_event_count",
            "user_event_count_before_cart",
            "log_user_event_count_before_cart",
            "item_event_count_before_cart",
            "log_item_event_count_before_cart",
            "missing_prior_pair_timing"
        ]
    ].head()
)

Number of model features created so far: 43


,prior_pair_event_count,log_prior_pair_event_count,user_event_count_before_cart,log_user_event_count_before_cart,item_event_count_before_cart,log_item_event_count_before_cart,missing_prior_pair_timing
0,0,0.000,0,0.000,0,0.000,1
1,0,0.000,0,0.000,0,0.000,1
2,1,0.693,2,1.099,1,0.693,0
3,0,0.000,0,0.000,0,0.000,1
4,1,0.693,1,0.693,1,0.693,0


In [18]:
# Purpose: Select leakage-safe model features and save the final cart-to-purchase modeling dataset.

model_feature_columns = [
    "has_prior_pair_interaction",
    "missing_prior_pair_timing",
    "log_prior_pair_event_count",
    "log_prior_pair_view_count",
    "log_prior_pair_transaction_count",
    "log_user_event_count_before_cart",
    "log_user_view_count_before_cart",
    "log_user_addtocart_count_before_cart",
    "log_user_transaction_count_before_cart",
    "log_item_event_count_before_cart",
    "log_item_view_count_before_cart",
    "log_item_addtocart_count_before_cart",
    "log_item_transaction_count_before_cart",
    "user_prior_purchase_share",
    "user_prior_cart_share",
    "item_prior_purchase_share",
    "item_prior_cart_share",
    "prior_pair_view_share",
    "prior_pair_recency_minutes",
    "prior_pair_interaction_span_minutes",
    "cart_hour",
    "cart_dayofweek",
    "cart_month"
]

modeling_dataset = cart_feature_table[
    [
        "visitorid",
        "itemid",
        "cart_time",
        *model_feature_columns,
        "purchased_within_14_days"
    ]
].copy()

feature_quality_summary = pd.DataFrame({
    "feature": model_feature_columns,
    "missing_count": [
        modeling_dataset[feature].isna().sum()
        for feature in model_feature_columns
    ],
    "unique_values": [
        modeling_dataset[feature].nunique()
        for feature in model_feature_columns
    ]
})

display(feature_quality_summary)

modeling_dataset.to_csv(
    processed_data_dir / "cart_purchase_modeling_dataset.csv",
    index=False
)

print(f"\nFinal modeling dataset shape: {modeling_dataset.shape}")
print(
    f"Positive purchase rate: "
    f"{modeling_dataset['purchased_within_14_days'].mean() * 100:.2f}%"
)

,feature,missing_count,unique_values
0,has_prior_pair_interaction,0,2
1,missing_prior_pair_timing,0,2
2,log_prior_pair_event_count,0,30
3,log_prior_pair_view_count,0,31
4,log_prior_pair_transaction_count,0,5
5,log_user_event_count_before_cart,0,2302
6,log_user_view_count_before_cart,0,2049
7,log_user_addtocart_count_before_cart,0,605
8,log_user_transaction_count_before_cart,0,459
9,log_item_event_count_before_cart,0,1215



Final modeling dataset shape: (56385, 27)
Positive purchase rate: 30.70%


In [19]:
# Purpose: Save feature definitions and quality checks for reproducibility and article documentation.

feature_documentation = pd.DataFrame({
    "feature": model_feature_columns,
    "feature_group": [
        "User-item history", "User-item history", "User-item history",
        "User-item history", "User-item history",
        "User history", "User history", "User history", "User history",
        "Item history", "Item history", "Item history", "Item history",
        "User history", "User history", "Item history", "Item history",
        "User-item history", "User-item history", "User-item history",
        "Temporal", "Temporal", "Temporal"
    ]
})

feature_documentation = feature_documentation.merge(
    feature_quality_summary,
    on="feature",
    how="left"
)

feature_documentation.to_csv(
    tables_dir / "05_model_feature_documentation.csv",
    index=False
)

print("Saved feature documentation and quality checks.")
display(feature_documentation)

Saved feature documentation and quality checks.


,feature,feature_group,missing_count,unique_values
0,has_prior_pair_interaction,User-item history,0,2
1,missing_prior_pair_timing,User-item history,0,2
2,log_prior_pair_event_count,User-item history,0,30
3,log_prior_pair_view_count,User-item history,0,31
4,log_prior_pair_transaction_count,User-item history,0,5
5,log_user_event_count_before_cart,User history,0,2302
6,log_user_view_count_before_cart,User history,0,2049
7,log_user_addtocart_count_before_cart,User history,0,605
8,log_user_transaction_count_before_cart,User history,0,459
9,log_item_event_count_before_cart,Item history,0,1215
